# Gaussian Discriminant Analysis

Datasets:
- https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Diabetes.csv

In [36]:
import sys
sys.path.append("..")

In [37]:
from impl.metrics import train_test_split

In [ ]:
from typing import Literal

import numpy as np
from numpy.typing import NDArray


class GaussianDiscriminantAnalysis:
    def __init__(self):
        self._n_features: int | None = None
        self._mu_0: NDArray | None = None
        self._mu_1: NDArray | None = None

        self._covariance: NDArray | None = None
        self._bernoulli: float| None = None

    def fit(self, X: NDArray, y: NDArray):
        n_features = X.shape[1]

        mu_0 = sum(int(y_i == 0) * x_i for x_i, y_i in zip(X, y))
        mu_1 = sum(int(y_i == 1) * x_i for x_i, y_i in zip(X, y))

        def _OP(x_i, mu):
            diff = x_i - mu
            diff_T = diff[np.newaxis].T
            return diff_T * diff
        
        covariance = sum(_OP(x_i, y_i) for x_i, y_i in zip(X, y)) / len(y)

        assert covariance.shape == (n_features, n_features) # type: ignore

        self._mu_0 = mu_0 # type: ignore
        self._mu_1 = mu_1 # type: ignore
        self._covariance = covariance # type: ignore
        self._bernoulli = np.sum(y) / len(y)
        self._n_features = n_features

    def _p_of_x_given_y(self, X: NDArray, y: Literal[0, 1]):
        # this uses log probabilities to ensure numerical stability. Works the same though.

        assert self._covariance is not None
        assert self._n_features is not None

        mu = self._mu_0 if y == 0 else self._mu_1
        covariance_inv = np.linalg.inv(self._covariance)
        diff = (X - mu)[np.newaxis]

        log_det = np.log(np.linalg.det(self._covariance))
        assert log_det >= 0

        comp1 = (-self._n_features/2) * np.log(2 * np.pi)
        comp2 = 0.5 * log_det

        comp3 = diff @ covariance_inv @ diff.T

        result = comp1 - comp2  - ((0.5 * np.log(np.e)) * comp3)
        return result[0, 0]
    
    def _p_y(self, y: Literal[0, 1]):
        assert self._bernoulli is not None

        return self._bernoulli if y == 1 else (1 - self._bernoulli)

    def predict(self, X: NDArray):
        p_x = (self._p_of_x_given_y(X, 1) * self._p_y(1)) + (self._p_of_x_given_y(X, 0) * self._p_y(0))
        p_x_y_1 = self._p_of_x_given_y(X, 1)
        p_x_y_0 = self._p_of_x_given_y(X, 0)

        p1 = (p_x_y_1 * self._p_y(1)) / p_x
        p0 = (p_x_y_0 * self._p_y(0)) / p_x

        if p1 >= p0:
            return 1
        else:
            return 0

In [39]:
import pandas as pd

df = pd.read_csv("../datasets/diabetes.csv")
X = df[["pregnancies","glucose","diastolic","triceps","insulin","bmi","dpf","age"]].to_numpy()
y = df["diabetes"].to_numpy()

In [40]:
X_train, y_train, X_test, y_test = train_test_split(X, y, 0.65)

In [41]:
model = GaussianDiscriminantAnalysis()
model.fit(X_train, y_train)

In [42]:
correct = 0
for idx, X_i in enumerate(X_test):
    prediction = model.predict(X_i)

    if prediction == y_test[idx]:
        correct += 1

print(f"{correct}/{len(X_test)} correct {(correct/len(X_test)) * 100}% accuracy")

183/269 correct 68.02973977695167% accuracy
